<a href="https://colab.research.google.com/github/sabithakrishnan/Tflite_demo/blob/main/TFlitewith_one_layer_neural_network_and_quantisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np

# Create simple training data for y = 2x - 1
x_train = np.array([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], dtype=float)
y_train = np.array([-3.0, -1.0, 1.0, 3.0, 5.0, 7.0], dtype=float)

# Define the model architecture
model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1, input_shape=[1])
])

# Compile and train
model.compile(optimizer='sgd', loss='mean_squared_error')
model.fit(x_train, y_train, epochs=500, verbose=0)

print("Model training complete.")

In [ ]:
import tensorflow as tf
import os

# Assuming 'model' is your trained Keras model from the previous step
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable the default optimization (Dynamic Range Quantization)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Convert and save
tflite_quant_model = converter.convert()
with open('model_quantized.tflite', 'wb') as f:
    f.write(tflite_quant_model)

print("Quantized model saved.")


In [ ]:
# Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path="simple_model.tflite")
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Prepare test input (e.g., x = 10.0)
test_val = np.array([[10.0]], dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], test_val)

# Run the inference
interpreter.invoke()

# Get the result (Expected: ~19.0)
prediction = interpreter.get_tensor(output_details[0]['index'])
print(f"Prediction for x=10: {prediction[0][0]:.4f}")
